In [1]:
#Imports
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt 
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report

In [2]:
#Loading data. (renamed csv files for simplicity)
#May need to edit path to where datais stored on your computer.
train = 'churn_training.csv'
#------------------------------------------------------------
train_df = pd.read_csv(train)
#Extras
analysis_df = train_df.copy()
pred_df = train_df.copy()


In [3]:
# ---------------- User input for subscription churn predictions

In [4]:
pred_df.dtypes

CustomerID           float64
Age                  float64
Gender                object
Tenure               float64
Usage Frequency      float64
Support Calls        float64
Payment Delay        float64
Subscription Type     object
Contract Length       object
Total Spend          float64
Last Interaction     float64
Churn                float64
dtype: object

In [5]:
pred_df = pred_df.drop(columns=['CustomerID'], errors='ignore')
pred_df.isnull().sum()

Age                  1
Gender               1
Tenure               1
Usage Frequency      1
Support Calls        1
Payment Delay        1
Subscription Type    1
Contract Length      1
Total Spend          1
Last Interaction     1
Churn                1
dtype: int64

In [6]:
pred_df = pred_df.dropna()
pred_df.isnull().sum()

Age                  0
Gender               0
Tenure               0
Usage Frequency      0
Support Calls        0
Payment Delay        0
Subscription Type    0
Contract Length      0
Total Spend          0
Last Interaction     0
Churn                0
dtype: int64

In [7]:
#Drops unimportant columns
pred_df = pred_df.dropna(subset=['Churn'])
#Converts columns to be numerical in preperation of executing machine classification models
pred_df['Churn'] = (
    pred_df['Churn']
    .replace({'Churned': 1, 'Active': 0})
    .astype(float)
    .astype(int)
)
pred_df['Gender'] = (pred_df['Gender'] == 'Female').astype(int)
#Identifying target and feature values
X = pred_df.drop('Churn', axis=1)
y = pred_df['Churn']
sub_mapping = {
    'Standard':0,
    'Premium':1,
    'Basic':2
}
pred_df['Subscription Type'] = pred_df['Subscription Type'].map(sub_mapping)

contract_mapping = {
    'Annual': 0,
    'Quarterly': 1,
    'Monthly': 2
}
pred_df['Contract Length'] = pred_df['Contract Length'].map(contract_mapping)

X_trn = pred_df.drop(columns=['Churn'])
y_trn = pred_df['Churn']

X_train, X_val, y_train, y_val = train_test_split(
    X_trn, y_trn, test_size=0.2, stratify=y_trn, random_state=42
)

In [8]:
from sklearn.calibration import CalibratedClassifierCV

X_train = X_train.dropna()
y_train = y_train[X_train.index]

X_val = X_val.dropna()
y_val = y_val[X_val.index]
gb = GradientBoostingClassifier(random_state=42)
#Improves model accuracy
calibrated_model = CalibratedClassifierCV(estimator=gb, method='isotonic', cv=5)
calibrated_model.fit(X_train, y_train)
y_pred = calibrated_model.predict(X_val)
y_proba = calibrated_model.predict_proba(X_val)

print("Calibrated Risk Percentages (First 5 rows):")
print(y_proba[:5])

print("\nAccuracy:", accuracy_score(y_val, y_pred))

Calibrated Risk Percentages (First 5 rows):
[[0.00000000e+00 1.00000000e+00]
 [1.00000000e+00 0.00000000e+00]
 [0.00000000e+00 1.00000000e+00]
 [1.00000000e+00 0.00000000e+00]
 [6.13359843e-04 9.99386640e-01]]

Accuracy: 0.9992514205995441


In [9]:
def predict_churn(user_input_dict):
    user_df = pd.DataFrame([user_input_dict])

    # Matches training columns exactly
    user_df = user_df.reindex(columns=X_train.columns, fill_value=0)

    prob = calibrated_model.predict_proba(user_df)[0][1]

    return round(prob * 100, 2)

In [12]:
print("\n--- Customer Churn Prediction ---\n")

user = {
    "Age": float(input("Enter your age: ")),
    
    "Gender": 1 if input("Enter Sex (Female/Male): ").lower() == "female" else 0,
    
    "Tenure": float(input("How many months has it been since you subscribed? ")),
    
    "Usage Frequency": float(input("Throughout your total usage, how often have you found yourself using the platform? ")),
    
    "Support Calls": float(input("How many times have you needed to call support? (0 if none) ")),
    
    "Payment Delay": float(input("How many days have you delayed your payment to the subscription? (0 if none) ")),
    
    "Subscription Type": {
        "standard": 0,
        "premium": 1,
        "basic": 2
    }[input("Enter Subscription Type (Standard/Premium/Basic): ").lower()],
    
    "Contract Length": {
        "annual": 0,
        "quarterly": 1,
        "monthly": 2
    }[input("Enter Contract Length (Annual/Quarterly/Monthly): ").lower()],
    
    "Total Spend": float(input("Enter Total Spend ($): ")),
    
    "Last Interaction": float(input("Enter Days Since Last Interaction: "))
}



--- Customer Churn Prediction ---



In [13]:
prob = predict_churn(user)

print("\n--------------------------")
print(f"Churn Risk: {prob}%")

if prob > 80:
    print(f"We have calculated a high risk of churn. There is a {prob}% likelihood that this is hurting your finances. Ultimately, we recommend you should end the subscription.")
elif prob < 80 and prob > 51:
    print(f"According to our calculations, you have a {prob}% churn risk. You may be better off leaving the subscription.")
elif prob < 51:
    print(f"Overall, you have a relatively low churn risk at {prob}%. This indicates that you may be using your subscription reasonably enough to continue without hurting finances.")


--------------------------
Churn Risk: 0.13%
Overall, you have a relatively low churn risk at 0.13%. This indicates that you may be using your subscription reasonably enough to continue without hurting finances.


In [45]:
print(gb.feature_importances_)

[1.36707560e-01 1.11362462e-02 1.38770838e-03 1.51343438e-04
 4.64878847e-01 1.23270795e-01 0.00000000e+00 0.00000000e+00
 2.56587184e-01 5.88031595e-03]


In [46]:
print(list(X_train.columns))

['Age', 'Gender', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Subscription Type', 'Contract Length', 'Total Spend', 'Last Interaction']


In [47]:
for col, imp in zip(X_train.columns, gb.feature_importances_):
    print(col, imp)
    

Age 0.1367075598615189
Gender 0.011136246238746454
Tenure 0.0013877083823110063
Usage Frequency 0.00015134343776683476
Support Calls 0.46487884714558364
Payment Delay 0.12327079475542538
Subscription Type 0.0
Contract Length 0.0
Total Spend 0.25658718423124666
Last Interaction 0.00588031594740106


In [48]:
print(y_train.value_counts(normalize=True))

Churn
0    0.546237
1    0.453763
Name: proportion, dtype: float64


In [49]:
print(gb.predict_proba(X_train[:10]))

[[9.96186506e-01 3.81349407e-03]
 [9.97256654e-01 2.74334594e-03]
 [1.12068634e-02 9.88793137e-01]
 [9.95328451e-01 4.67154937e-03]
 [4.21418543e-05 9.99957858e-01]
 [9.97474810e-01 2.52519013e-03]
 [3.96845212e-03 9.96031548e-01]
 [9.97221487e-01 2.77851321e-03]
 [9.97221487e-01 2.77851321e-03]
 [9.97256654e-01 2.74334594e-03]]
